# Snowflake Cortex: Complete Guide

Snowflake Cortex is a suite of AI and ML services built natively into Snowflake. It provides access to large language models (LLMs), search, agents, and AI functions — all running inside Snowflake's security and governance perimeter.

**What you will learn in this notebook:**

1. Cortex Base Models — The LLMs available in Snowflake
2. Cortex AI Functions — SQL-callable AI operations (AI_COMPLETE, AI_EXTRACT, etc.)
3. Cortex Search Service — Hybrid vector + keyword search for RAG
4. Cortex Analyst — Natural language to SQL over structured data
5. Cortex Agents — Orchestration layer that combines all the above
6. Cortex Fine-tuning — Customize models with your data
7. MCP Server — Expose Snowflake AI tools to external clients via Model Context Protocol
8. External Agents — Connect Cortex Agents to third-party tools (Jira, Salesforce, etc.)

---

**Core Principles:**
- All models run inside Snowflake's secure boundary — your data never leaves
- Snowflake never uses your data to train models available to other customers
- Access is governed through standard RBAC (role-based access control)
- No infrastructure to manage — all services are fully managed

---
## 1. Cortex Base Models

Cortex Base Models are the large language models (LLMs) available for use in Snowflake. You do not host or manage these models — Snowflake provides them as a service.

**Key points:**
- Models from multiple providers: Anthropic (Claude), OpenAI (GPT), Google (Gemini), Meta (Llama), Mistral, DeepSeek
- Models run via **cross-region inference** — availability depends on your account's `CORTEX_ENABLED_CROSS_REGION` parameter
- Use `SHOW CORTEX BASE MODELS` to see which models are available to your account
- Models are used by: `AI_COMPLETE`, Cortex Agents, Fine-tuning, and other AI functions

**Model Categories:**
| Category | Examples | Use Case |
|----------|----------|----------|
| Large (high quality) | claude-opus-4-6, openai-gpt-5.2 | Complex reasoning, multi-step tasks |
| Medium (balanced) | claude-sonnet-4-5, openai-gpt-5-mini | General purpose, good quality/cost tradeoff |
| Small (fast/cheap) | claude-haiku-4-5, llama3.1-8b | High-volume, latency-sensitive tasks |
| Embedding | snowflake-arctic-embed-l-v2.0 | Vector search, similarity |

**Access Requirement:** `SNOWFLAKE.CORTEX_USER` database role (granted to PUBLIC by default)

In [ ]:
%%sql -r base_models
-- List all available Cortex base models in your account
SHOW CORTEX BASE MODELS;

---
## 2. Cortex AI Functions

Cortex AI Functions are SQL-callable functions that perform AI operations on your data. Call them in any SELECT, WHERE, or JOIN clause — just like regular SQL functions.

**All functions live under the `SNOWFLAKE.CORTEX` schema.**

### Function Reference

| Function | What It Does | Input |
|----------|-------------|-------|
| `AI_COMPLETE` | General-purpose text/image completion with any model | Text, images, documents |
| `AI_CLASSIFY` | Classifies input into categories you define | Text, images, documents |
| `AI_FILTER` | Returns TRUE/FALSE for filtering rows | Text, images |
| `AI_EXTRACT` | Extracts structured fields from unstructured input | Text, images, documents |
| `AI_SENTIMENT` | Returns sentiment score (-1 to 1) | Text |
| `AI_TRANSLATE` | Translates text between languages | Text |
| `AI_SUMMARIZE_AGG` | Summarizes text across multiple rows | Text (aggregate) |
| `AI_AGG` | Custom aggregation with a prompt | Text (aggregate) |
| `AI_EMBED` | Generates embedding vectors | Text |
| `AI_SIMILARITY` | Calculates similarity between two inputs | Text/embeddings |
| `AI_PARSE_DOCUMENT` | Extracts text/layout from documents (OCR) | PDF, images on stage |
| `AI_REDACT` | Removes PII from text | Text |
| `AI_TRANSCRIBE` | Transcribes audio/video to text | Audio/video on stage |
| `AI_COUNT_TOKENS` | Counts tokens for a given model | Text |

In [ ]:
%%sql -r ai_complete_example
-- AI_COMPLETE: General-purpose completion
-- This is the most versatile function — use any model for any generative task
SELECT SNOWFLAKE.CORTEX.AI_COMPLETE(
    'claude-sonnet-4-5',
    'Explain what a data warehouse is in 2 sentences.'
) AS response;

In [ ]:
%%sql -r sentiment_example
-- AI_SENTIMENT: Returns a score from -1 (negative) to 1 (positive)
SELECT 
    text,
    SNOWFLAKE.CORTEX.AI_SENTIMENT(text) AS sentiment_score
FROM (VALUES 
    ('This product is amazing and exceeded my expectations!'),
    ('The service was terrible and I want a refund.'),
    ('The package arrived on Tuesday as expected.')
) AS t(text);

In [ ]:
%%sql -r classify_example
-- AI_CLASSIFY: Classify text into categories you define
SELECT SNOWFLAKE.CORTEX.AI_CLASSIFY(
    'My laptop screen is cracked and I need a replacement',
    ['Hardware Issue', 'Software Issue', 'Billing Question', 'General Inquiry']
) AS classification;

In [ ]:
%%sql -r extract_example
-- AI_EXTRACT: Extract structured data from unstructured text
SELECT SNOWFLAKE.CORTEX.AI_EXTRACT(
    'Please contact John Smith at john.smith@acme.com or call 555-0123. He works at Acme Corp in New York.',
    ['person_name', 'email', 'phone', 'company', 'city']
) AS extracted_data;

In [ ]:
%%sql -r translate_example
-- AI_TRANSLATE: Translate text between languages
SELECT SNOWFLAKE.CORTEX.AI_TRANSLATE(
    'Snowflake Cortex makes AI accessible to every data team.',
    'en',
    'fr'
) AS french_translation;

In [ ]:
%%sql -r structured_output_example
-- AI_COMPLETE with structured JSON output
SELECT SNOWFLAKE.CORTEX.AI_COMPLETE(
    'claude-sonnet-4-5',
    'Extract the company name, founding year, and headquarters city from: Snowflake Inc. was founded in 2012 and is headquartered in Bozeman, Montana.',
    {
        'response_format': {
            'type': 'json',
            'schema': {
                'type': 'object',
                'properties': {
                    'company': {'type': 'string'},
                    'founded_year': {'type': 'integer'},
                    'headquarters': {'type': 'string'}
                }
            }
        }
    }
) AS structured_output;

---
## 3. Cortex Search Service

Cortex Search is a fully managed hybrid search engine (vector + keyword) for text data stored in Snowflake. It powers RAG (Retrieval-Augmented Generation) applications and enterprise search.

**What it does:**
- Takes a natural language query and returns the most relevant documents/rows
- Combines semantic understanding (vector search) with exact matching (keyword search)
- Automatically builds and maintains the search index
- Refreshes incrementally as source data changes

**How it works:**
1. You define a source query (SELECT statement) that produces text data
2. Snowflake builds an index using an embedding model (e.g., `snowflake-arctic-embed-l-v2.0`)
3. You query the service via REST API, Python SDK, or SQL
4. Results come back ranked by relevance with optional filtering

**Key Parameters:**
- `ON` — The text column to search
- `ATTRIBUTES` — Columns available for filtering (e.g., category, date)
- `TARGET_LAG` — How fresh the index should be (e.g., '1 hour', '1 day')
- `EMBEDDING_MODEL` — Which embedding model to use

**Use Cases:**
- RAG engine for chatbots and agents
- Document search (policies, contracts, manuals)
- Support ticket retrieval
- Knowledge base search

In [ ]:
%%sql -r create_articles_table
-- Step 1: Create a source table with text data
CREATE OR REPLACE TABLE cortex_demo.public.support_articles (
    article_id INT,
    title VARCHAR,
    content VARCHAR,
    category VARCHAR,
    last_updated DATE
);

-- Insert sample data
INSERT INTO cortex_demo.public.support_articles VALUES
(1, 'How to reset password', 'To reset your password, go to Settings > Security > Reset Password. Click the reset link sent to your email.', 'Account', '2024-01-15'),
(2, 'Billing cycle explained', 'Your billing cycle starts on the 1st of each month. Charges are calculated based on daily usage.', 'Billing', '2024-02-01'),
(3, 'Data sharing guide', 'To share data with another account, create a share object and add the desired databases, schemas, or tables.', 'Features', '2024-03-10');

In [ ]:
%%sql -r create_search_service
-- Step 2: Create a Cortex Search Service over the text data
CREATE OR REPLACE CORTEX SEARCH SERVICE cortex_demo.public.article_search_service
    ON content
    ATTRIBUTES category
    WAREHOUSE = COMPUTE_WH
    TARGET_LAG = '1 day'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (
        SELECT
            article_id,
            title,
            content,
            category,
            last_updated
        FROM cortex_demo.public.support_articles
    );

**Querying the Search Service (Python example):**
```python
import json
from snowflake.core import Root

root = Root(session)
search_service = (
    root.databases["cortex_demo"]
    .schemas["public"]
    .cortex_search_services["article_search_service"]
)

results = search_service.search(
    query="how do I change my password",
    columns=["title", "content", "category"],
    filter={"@eq": {"category": "Account"}},
    limit=5
)
print(json.dumps(results.results, indent=2))
```

---
## 4. Cortex Analyst

Cortex Analyst converts natural language questions into accurate SQL queries against your structured data. Business users ask questions in plain English; Cortex Analyst generates and executes SQL.

**How it works:**
1. You define a **Semantic View** — a YAML-based object that describes your data in business terms
2. Users ask natural language questions via the REST API or Cortex Agents
3. Cortex Analyst generates SQL, respecting your semantic definitions and RBAC
4. Results are returned as query output

**Semantic View Components:**

| Component | What It Defines | Example |
|-----------|----------------|--------|
| Logical Tables | Business entities | `customers`, `orders`, `products` |
| Dimensions | Categorical columns | `customer_name`, `region`, `product_category` |
| Time Dimensions | Date/time columns | `order_date`, `created_at` |
| Facts | Row-level numeric values | `order_amount`, `quantity` |
| Metrics | Aggregated KPIs (with formulas) | `total_revenue = SUM(order_amount)` |
| Relationships | How tables join | `orders.customer_id = customers.id` |

**Key Features:**
- **Verified Query Repository (VQR)** — Gold-standard question/SQL pairs that improve accuracy
- **Custom Instructions** — Natural language rules (e.g., "revenue always excludes returns")
- **Multi-view routing** — Agent can pick the right semantic view automatically
- **Full RBAC** — Generated SQL respects all data access controls

In [ ]:
%%sql -r create_semantic_view
-- Example: Create a Semantic View for a revenue dataset
-- Semantic Views are defined in YAML and created via SQL

CREATE OR REPLACE SEMANTIC VIEW cortex_demo.public.revenue_semantic_view
  COMMENT = 'Revenue analysis semantic view'
  AS SEMANTIC VIEW SPEC $$
name: Revenue Analysis
description: Semantic view for analyzing revenue by product, region, and time.

tables:
  - name: orders
    base_table:
      database: cortex_demo
      schema: public
      table: orders
    dimensions:
      - name: order_id
        expr: order_id
        description: Unique order identifier
      - name: region
        expr: region
        description: Sales region
      - name: product_category
        expr: product_category
        description: Product category
    time_dimensions:
      - name: order_date
        expr: order_date
        description: Date the order was placed
    facts:
      - name: order_amount
        expr: order_amount
        description: Dollar amount of the order
      - name: quantity
        expr: quantity
        description: Number of units ordered
    metrics:
      - name: total_revenue
        expr: SUM(order_amount)
        description: Total revenue
      - name: average_order_value
        expr: AVG(order_amount)
        description: Average order value
      - name: total_orders
        expr: COUNT(DISTINCT order_id)
        description: Number of distinct orders
$$;

**Querying with Cortex Analyst (via REST or Python):**
```python
from snowflake.core import Root

root = Root(session)
analyst = root.databases["cortex_demo"].schemas["public"].semantic_views["revenue_semantic_view"]

# Ask a natural language question
response = analyst.message(
    prompt="What was total revenue by region last quarter?"
)
print(response.content)  # Contains generated SQL and results
```

**Via Cortex CLI:**
```bash
cortex analyst query "What are the top 5 products by revenue?" --view=cortex_demo.public.revenue_semantic_view
```

---
## 5. Cortex Agents

Cortex Agents is the orchestration layer that brings everything together. An agent reasons over a request, plans the work, calls tools, and generates a response — all fully managed by Snowflake.

**The Agent Reasoning Loop:**
```
   User Question
        |
        v
  +-- PLAN --------+
  | Parse request   |
  | Select tools    |
  | Split subtasks  |
  +---------+------+
            |
            v
  +-- USE TOOLS ---+
  | Cortex Analyst  | --> SQL over structured data
  | Cortex Search   | --> Retrieve from documents
  | Code Execution  | --> Python sandbox
  | Custom Tools    | --> Your stored procs/UDFs
  | Web Search      | --> Public internet
  +---------+------+
            |
            v
  +-- REFLECT -----+
  | Evaluate results|
  | Need more info? | --> Loop back to PLAN
  | Done?           | --> Generate response
  +---------+------+
            |
            v
     Final Answer
```

**Available Tools:**
| Tool | Type | What It Does |
|------|------|-------------|
| Cortex Analyst | Built-in | SQL generation over semantic views |
| Cortex Search | Built-in | Retrieve from unstructured text |
| Code Execution | Built-in | Run Python in a secure sandbox |
| Data to Chart | Built-in | Generate visualizations |
| Web Search | Built-in | Real-time internet search |
| Custom Tools | User-defined | Stored procedures and UDFs |
| Agent Skills | User-defined | Modular instruction packages |
| MCP Connectors | External | Tools from Jira, Salesforce, GitHub, etc. |

**Key Concepts:**
- **Thread** — Persisted conversation context across multiple turns
- **Run** — A single request/response cycle
- **Model** — The LLM powering the agent (recommend `auto` for best quality)
- **Instructions** — Natural language guidance for agent behavior

In [ ]:
%%sql -r create_agent
-- Create a Cortex Agent that uses both structured and unstructured data
CREATE OR REPLACE CORTEX AGENT cortex_demo.public.business_analyst_agent
  FROM SPECIFICATION $$
  models:
    - auto
  orchestration:
    planning_instructions: |
      You are a business analyst that answers questions about company revenue
      and support documentation. Use Cortex Analyst for numeric/SQL questions
      and Cortex Search for document-based questions.
    response_instructions: |
      Always cite your sources. For numeric answers, include the SQL used.
      Be concise and actionable.
  tools:
    - tool_type: cortex_analyst
    - tool_type: cortex_search
    - tool_type: data_to_chart
  tool_resources:
    cortex_analyst:
      - semantic_view: cortex_demo.public.revenue_semantic_view
    cortex_search:
      - service: cortex_demo.public.article_search_service
  $$;

**Querying the Agent (Python via REST API):**
```python
import requests, json

url = "https://<account_url>/api/v2/cortex/agents/cortex_demo.public.business_analyst_agent:run"
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

payload = {
    "messages": [{"role": "user", "content": "What was our total revenue last month by region?"}],
    "thread_id": "optional-thread-id-for-multi-turn"
}

response = requests.post(url, headers=headers, json=payload)
print(response.json())
```

---
## 6. Cortex Fine-tuning

Fine-tuning lets you customize a base LLM for your specific tasks using your own data. Use it when prompt engineering alone doesn't achieve the accuracy you need.

**When to fine-tune:**
- Domain-specific terminology that generic models get wrong
- Consistent output formatting requirements
- Task-specific accuracy improvements (classification, extraction)
- Reducing token usage by teaching the model your patterns

**How it works:**
1. Prepare training data as `prompt` / `completion` pairs in a Snowflake table
2. Call `SNOWFLAKE.CORTEX.FINETUNE('CREATE', ...)` to start training
3. Use the resulting model with `AI_COMPLETE` or `AI_EXTRACT`

**Available Base Models for Fine-tuning:**
- `llama3-8b`, `llama3-70b`
- `llama3.1-8b`, `llama3.1-70b`
- `mistral-7b`, `mixtral-8x7b`
- `arctic-extract` (specialized for document extraction)

In [ ]:
%%sql -r finetune_example
-- Step 1: Create training data table
CREATE OR REPLACE TABLE cortex_demo.public.finetune_training_data (
    prompt VARCHAR,
    completion VARCHAR
);

-- Insert training examples (at least 100 recommended for good results)
INSERT INTO cortex_demo.public.finetune_training_data VALUES
('Classify this support ticket: My account is locked', '{"category": "account_access", "priority": "high"}'),
('Classify this support ticket: How do I export data?', '{"category": "feature_question", "priority": "low"}'),
('Classify this support ticket: I was charged twice', '{"category": "billing", "priority": "high"}');

-- Step 2: Start fine-tuning job
-- SELECT SNOWFLAKE.CORTEX.FINETUNE(
--     'CREATE',
--     'my_custom_classifier',
--     'mistral-7b',
--     'SELECT prompt, completion FROM cortex_demo.public.finetune_training_data',
--     'SELECT prompt, completion FROM cortex_demo.public.finetune_validation_data'
-- );

-- Step 3: Check fine-tuning job status
-- SELECT SNOWFLAKE.CORTEX.FINETUNE('DESCRIBE', 'my_custom_classifier');

-- Step 4: Use the fine-tuned model
-- SELECT SNOWFLAKE.CORTEX.AI_COMPLETE(
--     'my_custom_classifier',
--     'Classify this support ticket: I cannot log in from my phone'
-- );

---
## 7. MCP Server (Model Context Protocol)

The Snowflake-managed MCP server lets external AI clients (Claude Desktop, LangGraph, custom apps) securely access your Snowflake AI tools without deploying separate infrastructure.

**What MCP does:**
- Provides a standardized interface for external AI tools to discover and invoke Snowflake resources
- Exposes Cortex Agents, Cortex Analyst, Cortex Search, and custom tools to any MCP-compatible client
- Handles authentication via Snowflake OAuth or External OAuth
- Enforces RBAC on every tool invocation

**Architecture:**
```
  External MCP Client              Snowflake
  (Claude Desktop,          ┌─────────────────────────┐
   LangGraph, etc.)         │  MCP Server Object      │
        │                   │  ┌───────────────────┐  │
        │── tools/list ────>│  │ Cortex Agent      │  │
        │                   │  │ Cortex Analyst    │  │
        │── tools/call ────>│  │ Cortex Search     │  │
        │                   │  │ SQL Execution     │  │
        │<── response ──────│  │ Custom UDFs/Procs │  │
        │                   │  └───────────────────┘  │
                            └─────────────────────────┘
```

**Supported Tool Types:**
| Type | What It Exposes |
|------|----------------|
| `CORTEX_AGENT_RUN` | A Cortex Agent (recommended as primary interface) |
| `CORTEX_ANALYST_MESSAGE` | Direct Cortex Analyst access |
| `CORTEX_SEARCH_SERVICE_QUERY` | Direct Cortex Search access |
| `SYSTEM_EXECUTE_SQL` | SQL query execution |
| `GENERIC` | Custom UDFs and stored procedures |

In [ ]:
%%sql -r create_mcp_server
-- Create an MCP Server that exposes a Cortex Agent to external clients
CREATE OR REPLACE MCP SERVER cortex_demo.public.my_mcp_server
  FROM SPECIFICATION $$
  tools:
    - title: "Business Data Agent"
      name: "business_data_agent"
      type: "CORTEX_AGENT_RUN"
      identifier: "cortex_demo.public.business_analyst_agent"
      description: "Answers business data questions using governed structured and unstructured data."

    - title: "Product Search"
      name: "product_search"
      type: "CORTEX_SEARCH_SERVICE_QUERY"
      identifier: "cortex_demo.public.article_search_service"
      description: "Search support articles and documentation."

    - title: "SQL Execution"
      name: "sql_exec"
      type: "SYSTEM_EXECUTE_SQL"
      description: "Execute read-only SQL queries."
      config:
        read_only: true
        query_timeout: 300
        warehouse: "COMPUTE_WH"
  $$;

**Connecting from an external MCP client (e.g., Claude Desktop):**

The MCP endpoint URL format is:
```
https://<account_url>/api/v2/databases/CORTEX_DEMO/schemas/PUBLIC/mcp-servers/MY_MCP_SERVER
```

Configure your MCP client to connect to this URL with Snowflake OAuth credentials.

---
## 8. External Agents (MCP Connectors)

MCP Connectors let your Cortex Agents connect to third-party services — Jira, Salesforce, GitHub, Glean, Linear — so agents can take action across enterprise systems, not just retrieve data.

**What this enables:**
- Create Jira tickets based on data insights
- Update Salesforce records from within a conversation
- Search GitHub repositories for code references
- Post to Slack channels
- Any action exposed by an external MCP server

**How it works:**
1. **Provider Setup** — Admin creates OAuth credentials on the third-party service
2. **API Integration** — Create a Snowflake API integration with OAuth credentials
3. **External MCP Server** — Create an external MCP server object referencing the integration
4. **Agent Configuration** — Add the external MCP server to a Cortex Agent
5. **User Authentication** — End users authenticate with the third-party service via OAuth

**Supported Connectors (built-in setup):**
- Atlassian (Jira & Confluence)
- GitHub
- Glean
- Linear
- Salesforce

**You can also build custom connectors** to any MCP-compatible server.

In [ ]:
%%sql -r create_external_mcp
-- Example: Connect a Cortex Agent to Jira via MCP Connector

-- Step 1: Create API integration for Atlassian
CREATE OR REPLACE API INTEGRATION jira_mcp_integration
    API_PROVIDER = external_mcp
    API_ALLOWED_PREFIXES = ('https://mcp.atlassian.com')
    API_USER_AUTHENTICATION = (
        TYPE = OAUTH_DYNAMIC_CLIENT,
        OAUTH_RESOURCE_URL = 'https://mcp.atlassian.com/v1/mcp'
    )
    ENABLED = TRUE;

-- Step 2: Create the External MCP Server object
CREATE OR REPLACE EXTERNAL MCP SERVER cortex_demo.public.jira_connector
    WITH DISPLAY_NAME = 'Atlassian (Jira & Confluence)'
    URL = 'https://mcp.atlassian.com/v1/mcp'
    API_INTEGRATION = jira_mcp_integration;

-- Step 3: Grant access to the connector
GRANT USAGE ON EXTERNAL MCP SERVER cortex_demo.public.jira_connector TO ROLE DATA_ENGINEER;
GRANT USAGE ON INTEGRATION jira_mcp_integration TO ROLE DATA_ENGINEER;

In [ ]:
%%sql -r create_ops_agent
-- Step 4: Create an Agent that uses the Jira connector
CREATE OR REPLACE CORTEX AGENT cortex_demo.public.ops_agent
  FROM SPECIFICATION $$
  models:
    - auto
  orchestration:
    planning_instructions: |
      You are an operations assistant. You can look up revenue data
      and create Jira tickets for follow-up actions.
    response_instructions: |
      When creating a Jira ticket, confirm the details before submitting.
  tools:
    - tool_type: cortex_analyst
    - tool_type: mcp_connector
  tool_resources:
    cortex_analyst:
      - semantic_view: cortex_demo.public.revenue_semantic_view
    mcp_connector:
      - mcp_server: cortex_demo.public.jira_connector
  $$;

---
## 9. How Everything Fits Together

```
┌─────────────────────────────────────────────────────────────────────────┐
│                         USER INTERFACES                                  │
│   Snowflake CoWork  |  Cortex Code  |  REST API  |  MCP Clients         │
└──────────────────────────────┬──────────────────────────────────────────┘
                               │
┌──────────────────────────────▼──────────────────────────────────────────┐
│                        CORTEX AGENTS                                     │
│   Plan → Use Tools → Reflect → Respond                                  │
│   (Orchestration layer — combines all tools below)                      │
└───┬──────────────┬──────────────┬──────────────┬───────────────┬────────┘
    │              │              │              │               │
    ▼              ▼              ▼              ▼               ▼
┌────────┐  ┌──────────┐  ┌──────────┐  ┌───────────┐  ┌──────────────┐
│Cortex  │  │ Cortex   │  │  Code    │  │  Custom   │  │   External   │
│Analyst │  │ Search   │  │Execution │  │  Tools    │  │ MCP Servers  │
│        │  │          │  │          │  │           │  │              │
│NL→SQL  │  │Vector +  │  │Python    │  │UDFs &     │  │Jira, GitHub  │
│over    │  │Keyword   │  │sandbox   │  │Stored     │  │Salesforce    │
│Semantic│  │Search    │  │          │  │Procedures │  │Glean, Linear │
│Views   │  │          │  │          │  │           │  │              │
└───┬────┘  └────┬─────┘  └──────────┘  └───────────┘  └──────────────┘
    │            │
    ▼            ▼
┌────────┐  ┌──────────┐
│Semantic│  │  Search  │
│Views   │  │ Services │
│(YAML)  │  │ (Index)  │
└───┬────┘  └────┬─────┘
    │            │
    ▼            ▼
┌────────────────────────────────────────┐
│         SNOWFLAKE DATA LAYER           │
│  Tables | Views | Stages | Documents   │
└────────────────────────────────────────┘
```

---

## Quick Reference: Access Control

| What You Need | Required Role / Privilege |
|---------------|-------------------------|
| Use any AI function | `SNOWFLAKE.CORTEX_USER` database role (PUBLIC by default) |
| Use Cortex Analyst only | `SNOWFLAKE.CORTEX_ANALYST_USER` database role |
| Use Cortex Agents only | `SNOWFLAKE.CORTEX_AGENT_USER` database role |
| Create a Search Service | `CREATE CORTEX SEARCH SERVICE` on schema |
| Create a Semantic View | `CREATE SEMANTIC VIEW` on schema |
| Create an Agent | `CREATE CORTEX AGENT` on schema |
| Create an MCP Server | `CREATE MCP SERVER` on schema |
| Fine-tune a model | `CREATE MODEL` on schema + `CORTEX_USER` role |

---

## Key Takeaways

| Component | One-Line Summary |
|-----------|------------------|
| **Base Models** | The LLMs available in Snowflake (Claude, GPT, Gemini, Llama, Mistral) |
| **AI Functions** | SQL-callable AI ops: classify, extract, translate, sentiment, complete |
| **Cortex Search** | Hybrid search engine for RAG and enterprise search |
| **Cortex Analyst** | Natural language → SQL via semantic views |
| **Cortex Agents** | Orchestration: plan, tool-use, reflect across all data types |
| **Fine-tuning** | Customize models with your training data |
| **MCP Server** | Expose Snowflake AI tools to external MCP clients |
| **External Agents** | Connect agents to third-party services (Jira, Salesforce, etc.) |